# Imidazolone & Thiazolone Library Generation

Virtual combinatorial library of COX-2 selective inhibitors derived from 
commercially available aldehydes, carboxylic acids and amines via 
reactions *in silico*.

**Pipeline:** Aldehydes + Carboxylic Acids → Erlenmeyer-Plöchl → Oxazolones → Aminolysis-GFPc → Imidazolones  
**Amines** feed into the Aminolysis-GFPc step as co-reactants.
**Parallel branch:** Oxazolones → Sulphur-Exchange → Thiazolones  


All three building-block sets pass Price → Veber filters before reactions.  
Final products (Imidazolones and Thiazolones) are filtered by Veber and Brenk+PAINS before export,  
and saved as **separate** CSV files.


In [1]:
from rdkit import Chem
import numpy as np
import pandas as pd

import py_utils as pu

print(f"py_utils v{pu.__version__}")

py_utils v3.15


## 📥 1. Load SDFs

The SDFs of the compounds provided by Enamine (with their IDs) that match a
specific substructure are loaded:
- Aldehydes, R-CHO
- Carboxylic compounds, R-COOH or RCOX
- Primary amines, R-NH₂

In [2]:
sdf_path_aldehydes   = 'mol_files/1. Enamine SDFs/TESTER_Aldehydes_120.sdf'
sdf_path_carboxylics = 'mol_files/1. Enamine SDFs/TESTER_CarboxylicAcids_750.sdf'
sdf_path_amines      = 'mol_files/1. Enamine SDFs/TESTER_PrimaryAmines_680.sdf'

df_aldehydes_raw   = pu.sdf_to_dataframe(sdf_path_aldehydes,   id_prefix='A')
df_carboxylics_raw = pu.sdf_to_dataframe(sdf_path_carboxylics, id_prefix='C')
df_amines_raw      = pu.sdf_to_dataframe(sdf_path_amines,      id_prefix='N') 

pu.report_df_size(df_aldehydes_raw,   'Aldehydes - Raw')
pu.report_df_size(df_carboxylics_raw, 'Carboxylics - Raw')
pu.report_df_size(df_amines_raw,      'Amines - Raw')

[SDF Reader] Loaded 120 unique compounds from mol_files/1. Enamine SDFs/TESTER_Aldehydes_120.sdf
[SDF Reader] Loaded 750 unique compounds from mol_files/1. Enamine SDFs/TESTER_CarboxylicAcids_750.sdf
[SDF Reader] Loaded 680 unique compounds from mol_files/1. Enamine SDFs/TESTER_PrimaryAmines_680.sdf
[Aldehydes - Raw] 120 rows
[Carboxylics - Raw] 750 rows
[Amines - Raw] 680 rows


## 🔸 2. Price filtration

Query the Enamine Store API for real-time pricing.  Compounds exceeding the
price thresholds are discarded during retrieval (no wasted work).

In [3]:
client = pu.EnamineClient()

print('=== Querying Enamine API for Aldehydes ===')
df_aldehydes_cheap = pu.add_enamine_prices(df_aldehydes_raw, client=client)

print('\n=== Querying Enamine API for Carboxylic Acids ===')
df_carboxylics_cheap = pu.add_enamine_prices(df_carboxylics_raw, client=client)

print('\n=== Querying Enamine API for Amines ===')
df_amines_cheap = pu.add_enamine_prices(df_amines_raw, client=client)

print('\n' + '='*50)
print('PRICING SUMMARY')
print('='*50)
pu.report_df_size(df_aldehydes_cheap,   'Aldehydes - Cheap')
pu.report_df_size(df_carboxylics_cheap, 'Carboxylics - Cheap')
pu.report_df_size(df_amines_cheap,      'Amines - Cheap')

=== Querying Enamine API for Aldehydes ===
[Enamine Pricing] Loaded 385 valid + 1165 invalid cached
[Enamine Pricing] Processing 120 compounds...
[Enamine Pricing] Using cache for 120 compounds
[Enamine Pricing] Querying 0 compounds via API
[Enamine Pricing] Filters: <= 40.0 EUR/g, <= 250.0 EUR/pack
[Enamine Pricing] Completed: 36/120 with valid pricing
⚠️  Removed 84 compounds (no valid pricing)

=== Querying Enamine API for Carboxylic Acids ===
[Enamine Pricing] Loaded 385 valid + 1165 invalid cached
[Enamine Pricing] Processing 750 compounds...
[Enamine Pricing] Using cache for 750 compounds
[Enamine Pricing] Querying 0 compounds via API
[Enamine Pricing] Filters: <= 40.0 EUR/g, <= 250.0 EUR/pack
[Enamine Pricing] Completed: 146/750 with valid pricing
⚠️  Removed 604 compounds (no valid pricing)

=== Querying Enamine API for Amines ===
[Enamine Pricing] Loaded 385 valid + 1165 invalid cached
[Enamine Pricing] Processing 680 compounds...
[Enamine Pricing] Using cache for 680 compound

## 🔸 3. Veber filtering (of building blocks)

Compute RDKit molecular descriptors, then apply Veber bioavailability criteria.
Adjust `VEBER_BB` limits as needed.

In [4]:
VEBER_BB = dict(max_tPSA=90, max_RotB=10, max_LogP=3)

df_aldehydes_druglike, df_aldehydes_rej = pu.filter_Veber(
    pu.add_rdkit_properties(df_aldehydes_cheap), **VEBER_BB,
)
df_carboxylics_druglike, df_carboxylics_rej = pu.filter_Veber(
    pu.add_rdkit_properties(df_carboxylics_cheap), **VEBER_BB,
)
df_amines_druglike, df_amines_rej = pu.filter_Veber(
    pu.add_rdkit_properties(df_amines_cheap), **VEBER_BB,
)

pu.report_df_size(df_aldehydes_druglike,   'Aldehydes - Druglike')
pu.report_df_size(df_carboxylics_druglike, 'Carboxylics - Druglike')
pu.report_df_size(df_amines_druglike,      'Amines - Druglike')

# Save Veber-filtered building blocks
pu.save_dataframe_as_csv(df_aldehydes_druglike,   'mol_files/2. Building Blocks/Aldehydes_druglike')
pu.save_dataframe_as_csv(df_aldehydes_rej,        'mol_files/2. Building Blocks/Rejected/Aldehydes_rejected_veber')
pu.save_dataframe_as_csv(df_carboxylics_druglike, 'mol_files/2. Building Blocks/Carboxylics_druglike')
pu.save_dataframe_as_csv(df_carboxylics_rej,      'mol_files/2. Building Blocks/Rejected/Carboxylics_rejected_veber')
pu.save_dataframe_as_csv(df_amines_druglike,      'mol_files/2. Building Blocks/Amines_druglike')
pu.save_dataframe_as_csv(df_amines_rej,           'mol_files/2. Building Blocks/Rejected/Amines_rejected_veber')

[filter_Veber] 29/36 accepted (80.6%), 7 rejected
[filter_Veber] 117/146 accepted (80.1%), 29 rejected
[filter_Veber] 179/203 accepted (88.2%), 24 rejected
[Aldehydes - Druglike] 29 rows
[Carboxylics - Druglike] 117 rows
[Amines - Druglike] 179 rows
Saved 29 compounds to: mol_files/2. Building Blocks/Aldehydes_druglike_29cmpds.csv
Saved 7 compounds to: mol_files/2. Building Blocks/Rejected/Aldehydes_rejected_veber_7cmpds.csv
Saved 117 compounds to: mol_files/2. Building Blocks/Carboxylics_druglike_117cmpds.csv
Saved 29 compounds to: mol_files/2. Building Blocks/Rejected/Carboxylics_rejected_veber_29cmpds.csv
Saved 179 compounds to: mol_files/2. Building Blocks/Amines_druglike_179cmpds.csv
Saved 24 compounds to: mol_files/2. Building Blocks/Rejected/Amines_rejected_veber_24cmpds.csv


## 🔸 4. Erlenmeyer-Plöchl reaction

Aldehydes + Carboxylic Acids → Oxazolones

In [5]:
df_oxazolones_raw = pu.rxn_ErlenmeyerPlochl(df_aldehydes_druglike, df_carboxylics_druglike)

pu.report_df_size(df_oxazolones_raw, 'Oxazolones - Raw')

[ErlenmeyerPlochl] 3,393 total pairs: 0 cache hits, 3,393 misses
[ErlenmeyerPlochl] Processing 3,393 misses with 7 workers (71 batches)...
[ErlenmeyerPlochl] Cache: 0 hits, 3393 misses (0.0% hit rate)
[ErlenmeyerPlochl] Stats: {'input_aldehydes': 29, 'input_carboxylics': 117, 'invalid_aldehyde': 0, 'invalid_carboxyl': 0, 'not_aldehyde': 0, 'not_carboxyl': 0, 'no_product': 0, 'problematic': 0, 'output_rows': 3393, 'cache_hits': 0, 'cache_misses': 3393}
[Oxazolones - Raw] 3393 rows


## 🔸 5. Veber filtering (of intermediates)

Compute descriptors and apply Veber criteria to oxazolones.

In [6]:
df_oxazolones_druglike, df_oxazolones_rej = pu.filter_Veber(
    pu.add_rdkit_properties(df_oxazolones_raw),
    max_tPSA=90, max_RotB=10, max_LogP=3,  # TODO: adjust for intermediates if needed
)

pu.report_df_size(df_oxazolones_druglike, 'Oxazolones - Druglike')

# Save intermediates
pu.save_dataframe_as_csv(df_oxazolones_druglike, 'mol_files/3. Oxazolones/Oxazolones')
pu.save_dataframe_as_csv(df_oxazolones_rej,      'mol_files/3. Oxazolones/Rejected/Oxazolones_rejected_veber')

[filter_Veber] 394/3393 accepted (11.6%), 2999 rejected
[Oxazolones - Druglike] 394 rows
Saved 394 compounds to: mol_files/3. Oxazolones/Oxazolones_394cmpds.csv
Saved 2999 compounds to: mol_files/3. Oxazolones/Rejected/Oxazolones_rejected_veber_2999cmpds.csv


## 🔹 6. Aminolysis-GFPc reaction

Oxazolones + Amines → Imidazolones

In [7]:
df_imidazolones_raw = pu.rxn_AminolysisGFPc(df_oxazolones_druglike, df_amines_druglike)

pu.report_df_size(df_imidazolones_raw, 'Imidazolones - Raw')

[AminolysisGFPc] 70,526 total pairs: 0 cache hits, 70,526 misses
[AminolysisGFPc] Processing 70,526 misses with 7 workers (71 batches)...
[AminolysisGFPc] Cache: 0 hits, 70526 misses (0.0% hit rate)
[AminolysisGFPc] Stats: {'input_oxazolones': 394, 'input_amines': 179, 'invalid_oxazolone': 0, 'invalid_amine': 0, 'not_oxazolone': 0, 'not_amine': 0, 'no_product': 0, 'problematic': 0, 'output_rows': 72890, 'cache_hits': 0, 'cache_misses': 70526}
[Imidazolones - Raw] 72496 rows


## 🔹 7. Sulphur-Exchange reaction

Oxazolones → Thiazolones

> **Note:** input is `df_oxazolones_druglike` (post-Veber, Cell 5) — not `df_imidazolones_raw` or `df_oxazolones_raw`.

In [8]:
THIOACETIC_PRICE_EQ = 10.0  # Price per equivalent of thioacetic acid

df_thiazolones_raw = pu.rxn_SulphurExchange(
    df_oxazolones_druglike,
    thioacetic_price_eq=THIOACETIC_PRICE_EQ,
)

pu.report_df_size(df_thiazolones_raw, 'Thiazolones - Raw')

[SulphurExchange] 394 valid oxazolones: 0 cache hits, 394 misses
[SulphurExchange] Processing 394 misses in main thread (< 1000)...
[SulphurExchange] Cache: 0 hits, 394 misses (0.0% hit rate)
[SulphurExchange] Stats: {'input_oxazolones': 394, 'thioacetic_price_eq': 10.0, 'invalid_oxazolone': 0, 'not_oxazolone': 0, 'no_product': 0, 'problematic': 0, 'output_rows': 394, 'cache_hits': 0, 'cache_misses': 394}
[Thiazolones - Raw] 394 rows


## 🔸 8. Veber filtering (of products)

Compute descriptors and apply Veber criteria to both product families.
Adjust `VEBER_PRODUCTS` limits independently from building blocks.

In [9]:
VEBER_PRODUCTS = dict(max_tPSA=90, max_RotB=10, max_LogP=3)  # TODO: adjust limits for products

df_imidazolones_druglike, df_imidazolones_rej = pu.filter_Veber(
    pu.add_rdkit_properties(df_imidazolones_raw), **VEBER_PRODUCTS,
)

df_thiazolones_druglike, df_thiazolones_rej = pu.filter_Veber(
    pu.add_rdkit_properties(df_thiazolones_raw), **VEBER_PRODUCTS,
)

pu.report_df_size(df_imidazolones_druglike, 'Imidazolones - Druglike')
pu.report_df_size(df_thiazolones_druglike,  'Thiazolones - Druglike')

[filter_Veber] 2842/72496 accepted (3.9%), 69654 rejected
[filter_Veber] 41/394 accepted (10.4%), 353 rejected
[Imidazolones - Druglike] 2842 rows
[Thiazolones - Druglike] 41 rows


## 🔸 9. Brenk + PAINS filters

Remove compounds containing Brenk structural alerts or PAINS substructures from products.


In [10]:
df_imidazolones_untoxic, df_imidazolones_rej = pu.filter_BrenkPAINS(df_imidazolones_druglike)
df_thiazolones_untoxic, df_thiazolones_rej = pu.filter_BrenkPAINS(df_thiazolones_druglike)

pu.report_df_size(df_imidazolones_untoxic, 'Imidazolones - Untoxic')
pu.report_df_size(df_thiazolones_untoxic,  'Thiazolones - Untoxic')

# Save rejected for analysis
pu.save_dataframe_as_csv(df_imidazolones_rej, 'mol_files/4. Imidazolones/Rejected/Imidazolones_rejected_brenkpains')
pu.save_dataframe_as_csv(df_thiazolones_rej,  'mol_files/5. Thiazolones/Rejected/Thiazolones_rejected_brenkpains')


[filter_BrenkPAINS] 2143/2842 accepted (75.4%)
[filter_BrenkPAINS] Rejected: Brenk=368, PAINS=331
[filter_BrenkPAINS] 31/41 accepted (75.6%)
[filter_BrenkPAINS] Rejected: Brenk=5, PAINS=5
[Imidazolones - Untoxic] 2143 rows
[Thiazolones - Untoxic] 31 rows
Saved 699 compounds to: mol_files/4. Imidazolones/Rejected/Imidazolones_rejected_brenkpains_699cmpds.csv
Saved 10 compounds to: mol_files/5. Thiazolones/Rejected/Thiazolones_rejected_brenkpains_10cmpds.csv


## 📤 10. Save CSVs

Imidazolones and Thiazolones saved as **separate** CSV files.

In [11]:
# TODO: define columns
# df_imidazolones_untoxic and df_thiazolones_untoxic must be saved SEPARATELY
pu.save_dataframe_as_csv(df_imidazolones_untoxic, 'mol_files/4. Imidazolones/Imidazolones')
pu.save_dataframe_as_csv(df_thiazolones_untoxic,  'mol_files/5. Thiazolones/Thiazolones')

Saved 2143 compounds to: mol_files/4. Imidazolones/Imidazolones_2143cmpds.csv
Saved 31 compounds to: mol_files/5. Thiazolones/Thiazolones_31cmpds.csv
